In [1]:
import torch

import torch.nn as nn

import torch.nn.functional as F

print(f"Pytorch version: {torch.__version__}")

Pytorch version: 2.7.1+cpu


In [2]:
import math

class PositionalEncoding(nn.Module):

    def __init__(self,d_model, max_len=5000):

        super(PositionalEncoding, self).__init__()

        pe=torch.zeros(max_len, d_model)

        position=torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) 

        div_term=torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))

        pe=pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):

        x=x+self.pe[:, :x.size(1)]

        return x 

In [3]:
batch_size=2

seq_len=10

d_model=512

x=torch.zeros(batch_size, seq_len, d_model)

pe=PositionalEncoding(d_model)

x_pe=pe(x)

print(x_pe.shape)

torch.Size([2, 10, 512])


In [4]:
class ScaledDotProductAttention(nn.Module):

    def __init__(self, dropout=0.1):

        super(ScaledDotProductAttention, self).__init__()

        self.dropout = nn.Dropout(dropout)

        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, Q, K, V, mask=None):

        d_k = Q.size(-1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        
        if mask is not None:

            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = self.softmax(scores)

        attn = self.dropout(attn)
        
        output = torch.matmul(attn, V)
        
        return output, attn


In [5]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads, dropout=0.1):

        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads==0, "d_model must be divisible by num_heads"

        self.d_k=d_model//num_heads

        self.num_heads=num_heads

        self.W_q=nn.Linear(d_model, d_model)

        self.W_k=nn.Linear(d_model, d_model)

        self.W_v=nn.Linear(d_model, d_model)

        self.fc=nn.Linear(d_model, d_model)

        self.attention=ScaledDotProductAttention(dropout)

        self.dropout=nn.Dropout(dropout)

        self.layer_norm=nn.LayerNorm(d_model)

    def forward(self, Q, K, V, mask=None):
        residual = Q

        batch_size = Q.size(0)

        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        context, attn = self.attention(Q, K, V, mask)

        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)

        output = self.fc(context)

        output = self.dropout(output)

        output = self.layer_norm(output + residual)

        return output, attn


In [6]:
batch_size=2

seq_len=10

d_model=512

num_heads=8

x=torch.rand(batch_size, seq_len, d_model)

mha=MultiHeadAttention(d_model, num_heads)

output, attn=mha(x, x, x)

print(output.shape)

print(attn.shape)

torch.Size([2, 10, 512])
torch.Size([2, 8, 10, 10])


In [7]:
class PositionwiseFeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout=0.1):

        super(PositionwiseFeedForward, self).__init__()

        self.linear1=nn.Linear(d_model, d_ff)

        self.linear2=nn.Linear(d_ff, d_model)

        self.dropout=nn.Dropout(dropout)

        self.layer_norm=nn.LayerNorm(d_model)

    def forward(self, x):

        residual=x
        
        x=self.linear1(x)

        x=torch.relu(x)

        x=self.dropout(x)

        x=self.linear2(x)

        x=self.dropout(x)

        x=self.layer_norm(x + residual)

        return x



In [8]:
ffn=PositionwiseFeedForward(d_model=512, d_ff=2048)

x=torch.rand(2, 10, 512)

output=ffn(x)

print(output.shape)

torch.Size([2, 10, 512])


In [9]:
class EncoderLayer(nn.Module):

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):

        super(EncoderLayer, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)

        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)  

    def forward(self, x, mask=None):

        x, attn = self.self_attn(x, x, x, mask)

        x = self.feed_forward(x)  

        return x, attn


In [10]:
encoder_layer=EncoderLayer(d_model=512, num_heads=8, d_ff=2048)

x=torch.rand(2, 10, 512)

output, attn=encoder_layer(x)

print(output.shape)

print(attn.shape)

torch.Size([2, 10, 512])
torch.Size([2, 8, 10, 10])


In [11]:
class Encoder(nn.Module):

    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1, max_len=512):

        super(Encoder, self).__init__()

        self.embedding=nn.Embedding(vocab_size, d_model)

        self.pos_encoding=PositionalEncoding(d_model, max_len)

        self.layers=nn.ModuleList([

            EncoderLayer(d_model, num_heads, d_ff, dropout)

            for _ in range(num_layers)

        ])

        self.dropout=nn.Dropout(dropout)

    def forward(self, src, mask=None):

        x=self.embedding(src)

        x=self.pos_encoding(x)

        x=self.dropout(x)

        attentions=[]

        for layer in self.layers:

            x, attn=layer(x, mask)

            attentions.append(attn)

        return x, attentions

In [12]:
vocab_size=10000

d_model=512

num_heads=8

d_ff=2048

num_layers=6

seq_len=10

batch_size=2

encoder=Encoder(vocab_size, d_model, num_heads, d_ff, num_layers)

src=torch.randint(0, vocab_size, (batch_size, seq_len))

output, attentions=encoder(src)

print("Output shape:", output.shape)

print("Attention shape:", attentions[0].shape)

print("Num of layers:", len(attentions))

Output shape: torch.Size([2, 10, 512])
Attention shape: torch.Size([2, 8, 10, 10])
Num of layers: 6


In [13]:
class DecoderLayer(nn.Module):

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):

        super(DecoderLayer, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)

        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
    
    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
       
        x, self_attn = self.self_attn(x, x, x, tgt_mask)

        x, cross_attn = self.cross_attn(x, enc_output, enc_output, src_mask)

        x = self.feed_forward(x)

        return x, self_attn, cross_attn


In [14]:
class Decoder(nn.Module):

    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1, max_len=512):

        super(Decoder, self).__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.pos_encoding = PositionalEncoding(d_model, max_len)

        self.layers = nn.ModuleList([

            DecoderLayer(d_model, num_heads, d_ff, dropout)

            for _ in range(num_layers)

        ])

        self.dropout = nn.Dropout(dropout)
    
    def forward(self, tgt, enc_output, src_mask=None, tgt_mask=None):

        x = self.embedding(tgt)

        x = self.pos_encoding(x)

        x = self.dropout(x)

        self_attns = []

        cross_attns = []

        for layer in self.layers:

            x, self_attn, cross_attn = layer(x, enc_output, src_mask, tgt_mask)
            
            self_attns.append(self_attn)

            cross_attns.append(cross_attn)

        return x, self_attns, cross_attns


In [15]:
vocab_size=10000

d_model=512

num_heads=8

d_ff=2048

num_layers=6

seq_len=10

batch_size=2

decoder=Decoder(vocab_size, d_model, num_heads, d_ff, num_layers)

tgt=torch.randint(0, vocab_size, (batch_size, seq_len))

enc_output=torch.rand(batch_size, seq_len, d_model)

output, self_attns, cross_attns=decoder(tgt, enc_output)

print("Output:", output.shape)

print("Self-Attn:", self_attns[0].shape)

print("Cross-Attns:", cross_attns[0].shape)

print("Layer:", len(self_attns))

Output: torch.Size([2, 10, 512])
Self-Attn: torch.Size([2, 8, 10, 10])
Cross-Attns: torch.Size([2, 8, 10, 10])
Layer: 6


In [16]:
import torch.nn as nn

class Transformer(nn.Module):

    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1, max_len=512):

        super(Transformer, self).__init__()

        self.encoder=Encoder(src_vocab_size, d_model, num_heads, d_ff, num_layers, dropout, max_len)

        self.decoder=Decoder(tgt_vocab_size, d_model, num_heads, d_ff, num_layers, dropout, max_len)

        self.out=nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):

        enc_output, enc_attns=self.encoder(src, src_mask)

        dec_output, self_attns, cross_attns=self.decoder(tgt, enc_output, src_mask, tgt_mask)

        output=self.out(dec_output)

        return output, enc_attns, self_attns, cross_attns

In [17]:
src_vocab_size = 10000

tgt_vocab_size = 10000

d_model = 512

num_heads = 8

d_ff = 2048

num_layers = 6

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers)

batch_size = 2

src_seq_len = 10

tgt_seq_len = 10

src = torch.randint(0, src_vocab_size, (batch_size, src_seq_len))

tgt = torch.randint(0, tgt_vocab_size, (batch_size, tgt_seq_len))

output, enc_attns, self_attns, cross_attns = transformer(src, tgt)

print("Final output logits:", output.shape)

print("Encoder attention:", enc_attns[0].shape)

print("Decoder self-attn:", self_attns[0].shape)

print("Decoder cross-attn:", cross_attns[0].shape)


Final output logits: torch.Size([2, 10, 10000])
Encoder attention: torch.Size([2, 8, 10, 10])
Decoder self-attn: torch.Size([2, 8, 10, 10])
Decoder cross-attn: torch.Size([2, 8, 10, 10])


In [18]:
import json

import re

with open("storytelling_dataset.json", "r", encoding="utf-8") as f:

    raw = f.read()

fixed = re.sub(r'}\s*{', '}\n{', raw.strip())

stories = []

for line in fixed.splitlines():

    try:

        stories.append(json.loads(line))

    except json.JSONDecodeError as e:

        print("Bad line:", line)

        print("Error:", e)

print(f"Loaded {len(stories)} stories")

print("Example:", stories[0])


Bad line: {"id": "sci-fi_tt_056", "title": "The Quantum Loop of Io's Abyss", "genre_tags": ["Time Travel", "Mystery", "Survival", "Space Colonization", "Thriller"], "word_count": 920, "text": "Dr. Jian Li, a geothermal engineer on humanity’s first deep-drill colony on Jupiter's moon Io, swore the active volcanoes were acting strangely. Not just their unpredictable eruptions, but the subtle, rhythmic 'pulses' her seismic sensors picked up – pulses that resonated with a faint, impossible temporal signature. It was a 'quantum loop', not geological, but artificial, emanating from deep within Io's scorching, sulfurous mantle. The mystery: who built it, and what was it looping?\n\nHer team found a colossal, alien structure, half-melted into the volcanic rock, humming with an unfathomable energy. It was a 'Temporal Wellspring', and it was cycling Io through rapid geological epochs. One moment, the surrounding rock was nascent magma; the next, hardened basalt; then, a shimmering, impossible cr

In [19]:
corpus_file = "lumos_corpus.txt"

with open(corpus_file, "w", encoding="utf-8") as f:

    for story in stories:

        f.write(story['text'].replace("\n", " ") + "\n")
        
        f.write(story['synopsis'].replace("\n", " ") + "\n")

print(f"Corpus file created: {corpus_file}")

Corpus file created: lumos_corpus.txt


In [20]:
with open("lumos_corpus.txt", "a", encoding="utf-8") as corpus:

    with open("Alice's Adventure.txt", "r", encoding="utf-8") as alice:

        corpus.write("\n" + alice.read().replace("\n", " ") + "\n")

    with open("Frankenstein; Or, The Modern Prometheus.txt", "r", encoding="utf-8") as frank:
        
        corpus.write("\n" + frank.read().replace("\n", " ") + "\n")

print("✅ Appended Alice's Adventure and Frankenstein to lumos_corpus.txt")

✅ Appended Alice's Adventure and Frankenstein to lumos_corpus.txt


In [21]:
import pandas as pd

df = pd.read_csv("stories.csv")

assert "content" in df.columns, "Column 'content' not found in CSV."

with open("lumos_corpus.txt", "r", encoding="utf-8") as f:

    before_lines = sum(1 for _ in f)

with open("lumos_corpus.txt", "a", encoding="utf-8") as corpus:

    for story in df["content"]:

        corpus.write(str(story).replace("\n", " ") + "\n")

with open("lumos_corpus.txt", "r", encoding="utf-8") as f:

    after_lines = sum(1 for _ in f)

added = after_lines - before_lines

print(f"✅ Total new stories added: {added}")

print(f"📘 Total lines in corpus now: {after_lines}")

print("\n🔹 Last 1 new entry added:")

with open("lumos_corpus.txt", "r", encoding="utf-8") as f:

    lines = f.readlines()

    for line in lines[-1:]:

        print(line.strip())


✅ Total new stories added: 1002
📘 Total lines in corpus now: 10130

🔹 Last 1 new entry added:
*** START OF THIS PROJECT GUTENBERG EBOOK THE PATRIOT ***          Produced by Greg Weeks, Mary Meehan and the Online  Distributed Proofreading Team at http://www.pgdp.net                                                  THE PATRIOT                            BY CHARLES L. FONTENAY                 _Earth was through with war. And while it is             right that man have peace, it is also right that           he have freedom. But Mars was in slavery, and to Mars          Cornel Lorensse dedicated his life and his talent...._               [Transcriber's Note: This etext was produced from                Worlds of If Science Fiction, August 1955.           Extensive research did not uncover any evidence that           the U.S. copyright on this publication was renewed.]      _The Martianne_ is heard occasionally these days as a stirring concert  or band selection. But there was a time when its

In [22]:
import sentencepiece as spm 

spm.SentencePieceTrainer.train(

    input='lumos_corpus.txt',

    model_prefix='lumos_tokenizer',

    vocab_size=19904, #Later we i will work on large dataset then i will increase it to 16k, 32k or even upto 50k 

    model_type='bpe',

    pad_id=0,

    unk_id=1,

    bos_id=2,

    eos_id=3,
)

In [23]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()

sp.load("lumos_tokenizer.model")

line_sci = "The Chronos Station imploded."

tokens_sci = sp.encode(line_sci, out_type=int)

decoded_sci = sp.decode(tokens_sci)

print("Sci-Fi Tokens:", tokens_sci)

print("Sci-Fi Decoded:", decoded_sci)

line_frank = "I beheld the wretch—the miserable monster whom I had created."

tokens_frank = sp.encode(line_frank, out_type=int)

decoded_frank = sp.decode(tokens_frank)

print("Frankenstein Tokens:", tokens_frank)

print("Frankenstein Decoded:", decoded_frank)

line_alice = "Curiouser and curiouser!' cried Alice."

tokens_alice = sp.encode(line_alice, out_type=int)

decoded_alice = sp.decode(tokens_alice)

print("Alice Tokens:", tokens_alice)

print("Alice Decoded:", decoded_alice)

line_book = "Earth was through with war, but Mars was still in chains."

tokens_book = sp.encode(line_book, out_type=int)

decoded_book = sp.decode(tokens_book)

print("BookCorpus Tokens:", tokens_book)

print("BookCorpus Decoded:", decoded_book)

Sci-Fi Tokens: [62, 482, 3719, 5071, 19852]
Sci-Fi Decoded: The Chronos Station imploded.
Frankenstein Tokens: [124, 125, 5, 385, 9, 19, 3197, 146, 19881, 1317, 42, 6552, 226, 1402, 6979, 232, 67, 124, 155, 1435, 19852]
Frankenstein Decoded: I beheld the wretch—the miserable monster whom I had created.
Alice Tokens: [6696, 12, 70, 677, 966, 12, 1, 19855, 5909, 2211, 294, 19852]
Alice Decoded: Curiouser and curiouser ⁇ ' cried Alice.
BookCorpus Tokens: [562, 53, 334, 143, 263, 19849, 156, 2010, 19839, 53, 2420, 41, 6048, 19852]
BookCorpus Decoded: Earth was through with war, but Mars was still in chains.


In [24]:
import json

existing_stories = []

with open("storytelling_dataset.json", "r", encoding="utf-8") as f:

    for line in f:

        try:

            existing_stories.append(json.loads(line))

        except json.JSONDecodeError:

            pass

with open("lumos_corpus.txt", "r", encoding="utf-8") as f:

    corpus_lines = f.readlines()

existing_texts = set(story['text'] for story in existing_stories)

new_entries = []

for line in corpus_lines:

    text = line.strip()

    if text and text not in existing_texts:

        new_entries.append({

            "text": text,

            "synopsis": "" 

        })

with open("storytelling_dataset.json", "a", encoding="utf-8") as f:

    for entry in new_entries:

        json.dump(entry, f)
        
        f.write("\n")

print(f"✅ Added {len(new_entries)} new stories to storytelling_dataset.json")


✅ Added 0 new stories to storytelling_dataset.json


In [25]:
from torch.utils.data import Dataset

import torch

class StoryDataset(Dataset):

    def __init__(self, stories, sp, max_len=512):

        self.stories = stories

        self.sp = sp

        self.max_len = max_len

    def __len__(self):

        return len(self.stories)

    def __getitem__(self, idx):

        for _ in range(len(self.stories)):

            story = self.stories[idx]

            src_text = story.get('text', '').strip()

            tgt_text = story.get('synopsis', '').strip()

            if not tgt_text:

                tgt_text = src_text

            if not src_text or not tgt_text:

                idx = (idx + 1) % len(self.stories)

                continue

            src_ids = self.sp.encode(src_text, out_type=int)

            tgt_ids = self.sp.encode(tgt_text, out_type=int)

            src_ids = [2] + src_ids[:self.max_len - 2] + [3]

            tgt_ids = [2] + tgt_ids[:self.max_len - 2] + [3]

            src_ids += [0] * (self.max_len - len(src_ids))

            tgt_ids += [0] * (self.max_len - len(tgt_ids))

            return torch.tensor(src_ids), torch.tensor(tgt_ids)

        dummy = [0] * self.max_len

        return torch.tensor(dummy), torch.tensor(dummy)



In [29]:
from torch.utils.data import DataLoader

dataset = StoryDataset(stories, sp, max_len=512)

valid_count = 0

for i in range(len(dataset)):

    try:
        src, tgt = dataset[i]

        valid_count += 1

    except Exception as e:

        print(f"Bad entry at {i}: {e}")

print(f"✅ Total valid entries: {valid_count}/{len(stories)}")

dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

✅ Total valid entries: 4562/4562


In [30]:
vocab_size=sp.get_piece_size()

print("Your custom tokenizer vocab size:", vocab_size)

Your custom tokenizer vocab size: 19904


In [31]:
transformer = Transformer(

    src_vocab_size=vocab_size,

    tgt_vocab_size=vocab_size,

    d_model=d_model,

    num_heads=num_heads,

    d_ff=d_ff,

    num_layers=num_layers
    
)


In [ ]:
import torch

import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transformer.to(device)

optimizer = torch.optim.Adam(transformer.parameters(), lr=1e-4)

loss_fn = nn.CrossEntropyLoss(ignore_index=0)

num_epochs = 3  

for epoch in range(num_epochs):

    transformer.train()

    start_time = time.time()

    total_loss = 0

    for src, tgt in dataloader:

        src, tgt = src.to(device), tgt.to(device)

        optimizer.zero_grad()

        output, *_ = transformer(src, tgt[:, :-1])  

        loss = loss_fn(output.view(-1, output.size(-1)), tgt[:, 1:].reshape(-1))

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"✅ Epoch {epoch + 1} completed in {time.time() - start_time:.2f}s - Avg Loss: {total_loss / len(dataloader):.4f}")


In [34]:
import torch

def sample_next_token(logits, temperature=1.0, top_k=10):

    logits = logits / temperature

    probs = torch.softmax(logits, dim=-1)

    if top_k > 0:

        top_k_probs, top_k_idx = torch.topk(probs, top_k)

        top_k_probs = top_k_probs / top_k_probs.sum()

        next_token = top_k_idx[torch.multinomial(top_k_probs, num_samples=1)].item()

    else:
        
        next_token = torch.multinomial(probs, num_samples=1).item()

    return next_token


In [35]:
import torch

def generate_summary(model, sp, src_text, max_len=50):

    model.eval()

    raw_src_ids = sp.encode(src_text, out_type=int)

    raw_src_ids = raw_src_ids[:510]

    src_ids = [2] + raw_src_ids + [3]

    src_tensor = torch.tensor([src_ids + [0] * (512 - len(src_ids))])

    enc_output, enc_attns = model.encoder(src_tensor)

    generated = [2]

    for _ in range(max_len):

        tgt_tensor = torch.tensor([generated + [0] * (512 - len(generated))])

        dec_output, _, _ = model.decoder(tgt_tensor, enc_output)

        logits = model.out(dec_output)

        if len(generated) < 15:

            logits[0, len(generated)-1, 3] = -float('inf')

        logits[0, len(generated)-1, 2] = -float('inf')

        if len(generated) >= 2:

            prev_token = generated[-1]

            logits[0, len(generated)-1, prev_token] = -float('inf')

        next_token = sample_next_token(

            logits[0, len(generated)-1],

            temperature=0.8,

            top_k=20

        )

        if next_token == 3 and len(generated) >= 15:

            break

        if next_token == 3 and len(generated) < 15:

            next_token = torch.randint(4, logits.size(-1), (1,)).item()

        generated.append(next_token)

    print("Generated tokens:", generated)

    decoded = sp.decode(generated[1:])
    
    return decoded


In [37]:
sample_story = stories[0]['text']  

summary = generate_summary(transformer, sp, sample_story)

print("Generated summary:", summary)

Generated tokens: [2, 1416, 121, 4062, 121, 52, 807, 6546, 2931, 365, 13961, 645, 466, 348, 185, 2693, 664, 1272, 45, 49, 673, 387, 471, 4405, 741, 3348, 31, 125, 3177, 1064, 1315, 1317, 172, 13952, 478, 1511, 4106, 1284, 2849, 258, 2849, 1355, 2535, 1842, 45, 2913, 1615, 1358, 4046, 1330, 13952]
Generated summary: deliberate on recorder on was trapped grows faces Unit- extract Regulators'. timeRecorder across Kaito ' of alien break futuresridden Drglacial to humanityundesirable colony stand expose He, systemGate quarantine stability manipulation him manipulation fightssplinters accidentally ' link maintainsurvivalrandomize enforcers,


In [39]:
torch.save(transformer.state_dict(), "lumos_transformer_epoch1.pth")
print("Model saved!")

Model saved!


In [40]:
transformer.load_state_dict(torch.load("lumos_transformer_epoch1.pth"))
transformer.eval()

Transformer(
  (encoder): Encoder(
    (embedding): Embedding(14000, 512)
    (pos_encoding): PositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (self_attn): MultiHeadAttention(
          (W_q): Linear(in_features=512, out_features=512, bias=True)
          (W_k): Linear(in_features=512, out_features=512, bias=True)
          (W_v): Linear(in_features=512, out_features=512, bias=True)
          (fc): Linear(in_features=512, out_features=512, bias=True)
          (attention): ScaledDotProductAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (dropout): Dropout(p=0.1, inplace=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        )
        (feed_forward): PositionwiseFeedForward(
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (dropo